In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

In [2]:
words = open('../names.txt').read().splitlines()

In [3]:
random.shuffle(words)

In [4]:
alph = ['.'] + sorted(set(''.join(words)))
A = len(alph)
stoi = {c: i for i, c in enumerate(alph)}
itos = {v: k for k, v in stoi.items()}

In [5]:
block_size = 3

def build_dataset(ws):
    X = []
    Y = []
    for w in ws:
        context = [0] * block_size
        for c in w + '.':
            X.append(context)
            Y.append(stoi[c])
            context = context[1:] + [stoi[c]]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:n2])
Xts, Yts = build_dataset(words[n2:])

In [6]:
class Linear:

    def __init__(self, fan_in, fan_out, bias = True):
        self.fan_in = fan_in
        self.fan_out = fan_out
        self.weights = torch.randn((fan_in, fan_out), generator = gen) / fan_in ** 0.5
        self.bias = None if bias is False else torch.zeros(fan_out)
    
    def __call__(self, x):
        self.out = x @ self.weights
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])

class Tanh:

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []

class BatchNorm1d:

    def __init__(self, fan_in, eps = 1e-5, momentum = 0.01):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.running_mean = torch.zeros(fan_in)
        self.running_var = torch.ones(fan_in)
        self.gain = torch.ones(fan_in)
        self.bias = torch.zeros(fan_in)
    
    def __call__(self, x):
        if self.training:
            bnmean = x.mean(axis = 0, keepdims = True)
            bnvar = x.var(axis = 0, keepdims = True)
            self.out = (x - bnmean) / torch.sqrt(bnvar + self.eps) * self.gain + self.bias
            with torch.no_grad():
                self.running_mean = self.running_mean * (1.0 - self.momentum) + bnmean * self.momentum
                self.running_var = self.running_var * (1.0 - self.momentum) + bnvar * self.momentum
        else:
            with torch.no_grad():
                self.out = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps) * self.gain + self.bias
        return self.out
    
    def parameters(self):
        return [self.gain, self.bias]
        

In [7]:
gen = torch.Generator().manual_seed(42)
embd_size = 10
layer_size = 300
C = torch.randn((A, embd_size), generator = gen)
model = [
    Linear(embd_size * block_size, layer_size, bias = False),
    BatchNorm1d(layer_size),
    Tanh(),
    Linear(layer_size, A, bias = True)

]
with torch.no_grad():
   model[-1].weights *= 0.01
parameters = []
for layer in model:
    parameters.extend(layer.parameters())
for p in parameters:
    p.requires_grad = True
print(f'Total number of parameters: {sum(p.nelement() for p in parameters)}')

Total number of parameters: 17727


In [8]:
batch_size = 32
epochs = 200000
for e in range(epochs+1):
    batch = torch.randint(0, len(Xtr), (batch_size, ), generator = gen)
    logits = C[Xtr[batch]].view(-1, block_size * embd_size)
    for layer in model:
        logits = layer(logits)
    loss = F.cross_entropy(logits, Ytr[batch])

    for p in parameters:
        p.grad = None

    loss.backward()
    lr = (0.1 if e < 100000 else 0.01)
    for p in parameters:
        p.data += -lr * p.grad
    if e%10000 == 0:
       print(f'Epoch {e}/{epochs}: loss: {loss.item()}')

Epoch 0/200000: loss: 3.2936439514160156
Epoch 10000/200000: loss: 1.9635767936706543
Epoch 20000/200000: loss: 2.5811121463775635
Epoch 30000/200000: loss: 2.461538314819336
Epoch 40000/200000: loss: 2.0577456951141357
Epoch 50000/200000: loss: 2.507392644882202
Epoch 60000/200000: loss: 2.5874128341674805
Epoch 70000/200000: loss: 2.1065123081207275
Epoch 80000/200000: loss: 1.9323517084121704
Epoch 90000/200000: loss: 2.281916618347168
Epoch 100000/200000: loss: 2.420305013656616
Epoch 110000/200000: loss: 2.30951189994812
Epoch 120000/200000: loss: 1.920082926750183
Epoch 130000/200000: loss: 1.7505080699920654
Epoch 140000/200000: loss: 2.437671422958374
Epoch 150000/200000: loss: 2.04647159576416
Epoch 160000/200000: loss: 1.7194150686264038
Epoch 170000/200000: loss: 2.143537998199463
Epoch 180000/200000: loss: 2.1214797496795654
Epoch 190000/200000: loss: 2.326050043106079
Epoch 200000/200000: loss: 2.3493146896362305


In [9]:
@torch.no_grad()
def getloss(X, Y):
    for layer in model:
        if isinstance(layer, BatchNorm1d):
            layer.training = False
    logits = C[X].view(-1, block_size * embd_size)
    for layer in model:
        logits = layer(logits)
    loss = F.cross_entropy(logits, Y)
    return loss.item()

In [10]:
getloss(Xtr, Ytr)

2.0765912532806396

In [11]:
getloss(Xval, Yval)

2.1176671981811523

In [12]:
getloss(Xts, Yts)

2.115943431854248